# AI Ignite at York - Forge Track
## Forge Workshop 01 - Emotion Detection - Base Notebook

**Workshop:** BiLSTM vs BERT vs LLMs on MELD  
**Date:** Friday 27 November 2026, WSC 101  
**Leads:** Sagar, Mehwish, Ali, Davit, Sheheryar  

> **How to use this notebook:**  
> Open it in Google Colab and run the cells from top to bottom. It clones the workshop repo, loads MELD with its **official train / dev / test split**, and defines how every model saves its predictions.  
> Each lead's notebook starts from the variables created here: `X_train`, `X_test`, `y_train`, `y_test`.

---

## Step 1 - Get the repo and create folders
In Colab this clones the workshop repo so `data/processed/` is available. Running locally from inside the repo, it just makes sure the folders exist.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sagarh12/forge-workshop.git"
REPO_DIR = "forge-workshop"

# In Colab the notebook starts in /content with no repo, so clone it once.
if "google.colab" in sys.modules and not os.path.exists("data/processed"):
    if not os.path.exists(REPO_DIR):
        subprocess.check_call(["git", "clone", "-q", REPO_URL])
    os.chdir(REPO_DIR)

folders = [
    "data/raw",
    "data/processed",
    "notebooks/davit",
    "notebooks/ali",
    "notebooks/sheheryar",
    "outputs/predictions",
    "outputs/reports",
    "outputs/figures",
    "slides",
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Working directory:", os.getcwd())
print("Folders ready:", ", ".join(folders))

## Step 2 - Install extra packages
Colab already has pandas, scikit-learn, matplotlib, seaborn, torch and transformers, so we only install what is missing. This takes about 20 seconds.

In [ ]:
%pip install -q imbalanced-learn google-genai
print("Extra packages installed")

## Step 3 - Imports
Shared imports used across the workshop. Note there is deliberately no `train_test_split` here: we use MELD's official split in Step 6.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")

print("All imports loaded successfully")

## Step 4 - Load MELD

**MELD** is a dataset of real TV-show dialogue, with each line labelled by humans with one of seven emotions.

> **Mehwish** prepares the cleaned file and commits it to `data/processed/meld_clean.csv`. Until it is committed this cell will stop with a clear message.

Columns: `utterance`, `label`, `dialogue_id`, `utterance_id`, `speaker`, `split`  
The seven labels (lowercase): `anger`, `disgust`, `fear`, `joy`, `neutral`, `sadness`, `surprise`.

In [ ]:
DATA_PATH = "data/processed/meld_clean.csv"
EXPECTED_COLUMNS = ["utterance", "label", "dialogue_id", "utterance_id", "speaker", "split"]
EXPECTED_LABELS = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"{DATA_PATH} is not in the repo yet. Mehwish is still cleaning the MELD files. "
        "Check back once she has committed it."
    )

meld = pd.read_csv(DATA_PATH)

missing = [c for c in EXPECTED_COLUMNS if c not in meld.columns]
assert not missing, f"meld_clean.csv is missing columns: {missing}"
assert meld["utterance"].notna().all(), "Some utterances are empty"
unknown = set(meld["label"].unique()) - EXPECTED_LABELS
assert not unknown, f"Unexpected label strings: {unknown} (labels must be lowercase and consistent)"

print(f"MELD loaded: {len(meld)} rows")
print(meld.head())
print("\nEmotion labels:", sorted(meld["label"].unique()))

## Step 5 - Class Distribution Check
Always check the class balance before training anything. MELD is heavily dominated by `neutral`, which is why accuracy alone will mislead us later.

In [ ]:
order = meld["label"].value_counts().index
fig, ax = plt.subplots(figsize=(9, 5))
sns.countplot(data=meld, x="label", order=order, color="#1424D0", ax=ax)
ax.set_title("MELD - Class Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Emotion")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("outputs/figures/class_distribution.png", dpi=150)
plt.show()
print((meld["label"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")

## Step 6 - Official Train / Dev / Test Split
**Do not use `train_test_split` anywhere in the workshop.** MELD ships with its own split. If one model is scored on different rows than another, the comparison is meaningless and nothing visibly breaks. Everyone uses this split.

In [ ]:
split_col = meld["split"].str.lower().replace({"val": "dev", "validation": "dev"})
train_df = meld[split_col == "train"].reset_index(drop=True)
dev_df   = meld[split_col == "dev"].reset_index(drop=True)
test_df  = meld[split_col == "test"].reset_index(drop=True)

assert len(train_df) > 0 and len(test_df) > 0, "Split column must contain 'train' and 'test' rows"

# MELD reuses dialogue ids across its official splits, so uniqueness is checked per (split, dialogue, utterance)
dupes = meld.duplicated(subset=["split", "dialogue_id", "utterance_id"]).sum()
assert dupes == 0, f"{dupes} duplicate (split, dialogue_id, utterance_id) rows found"

X_train, y_train = train_df["utterance"].tolist(), train_df["label"].tolist()
X_dev,   y_dev   = dev_df["utterance"].tolist(),   dev_df["label"].tolist()
X_test,  y_test  = test_df["utterance"].tolist(),  test_df["label"].tolist()

print(f"Train : {len(X_train)}")
print(f"Dev   : {len(X_dev)}")
print(f"Test  : {len(X_test)}")
print("\nUse X_train, X_test, y_train, y_test in your individual notebooks. Never re-split.")

## Step 7 - Results file format (used by Parts B, C and D)
You don't need to change anything here during Part A. This cell defines `save_predictions()`. Ali's and Davit's notebooks use it to save their results in one shared format, so Sheheryar's scoreboard can compare every model fairly. It checks each file against the official test set and refuses to save anything misaligned.

**Required columns:** `utterance`, `true_label`, `predicted_label`, `model_name`  
**Allowed model names:** `bilstm`, `bert_finetuned`, `zero_shot`, `few_shot`, `balanced_few_shot`

In [ ]:
ALLOWED_MODELS = {"bilstm", "bert_finetuned", "zero_shot", "few_shot", "balanced_few_shot"}

def save_predictions(X, y_true, y_pred, model_name: str, full_test_set: bool = True):
    """
    Save model predictions in the standard format.

    Args:
        X            : utterances, in the same order as the official test set
        y_true       : true labels
        y_pred       : predicted labels
        model_name   : one of ALLOWED_MODELS
        full_test_set: set False only for capped runs such as 100-row LLM demos
    """
    assert model_name in ALLOWED_MODELS, f"model_name must be one of {sorted(ALLOWED_MODELS)}"
    assert len(X) == len(y_true) == len(y_pred), "X, y_true and y_pred must be the same length"

    # Rows must be the official test rows, in the official order
    n = len(X)
    assert list(X) == X_test[:n], "Utterances do not match the official test set order. Did you re-split or shuffle?"
    assert list(y_true) == y_test[:n], "true_label does not match the official test labels"
    if full_test_set:
        assert n == len(X_test), f"Expected {len(X_test)} rows (full test set), got {n}"

    bad = set(y_pred) - EXPECTED_LABELS
    assert not bad, f"Predicted labels outside the seven emotions: {bad}"

    df = pd.DataFrame({
        "utterance":       list(X),
        "true_label":      list(y_true),
        "predicted_label": list(y_pred),
        "model_name":      model_name,
    })
    path = f"outputs/predictions/{model_name}_predictions.csv"
    df.to_csv(path, index=False)
    print(f"Predictions saved to {path} ({n} rows)")
    return df

# Example usage (replace with your real predictions):
# save_predictions(X_test, y_test, y_pred, model_name="bert_finetuned")

---
## Folder reference

| Folder | What's inside | Owner |
|---|---|---|
| `forge_workshop_base.ipynb` | Part A: this notebook (data and setup) | Sagar |
| `data/processed/` | `meld_clean.csv` and `meld_context.csv`, the cleaned MELD dataset | Mehwish |
| `notebooks/ali/` | Part B1: train BiLSTM and BERT. Part B2: see the results | Ali |
| `notebooks/davit/` | Part C: ask Gemini (zero-shot, few-shot, balanced) | Davit |
| `notebooks/sheheryar/` | Part D: the scoreboard | Sheheryar |
| `outputs/predictions/` | one results file per model | Ali and Davit |
| `outputs/figures/` | charts and training logs | Ali and Davit |
| `tests/` | automatic checks for the scoreboard | Sheheryar |